In [0]:
This notebook loads, cleans, transforms, and publishes London Transport datasets in Databricks.

## Part1 - setup

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
spark

## Part2 - Access raw files_

In [0]:
%python

raw_base_path = "/Volumes/workspace/london_transport_schema/london_transport_data"


- stations.csv
- lines.csv
- journeys.json
- vehicle_types.csv
- operators.csv
- zones.csv
- disruptions.json
- fares.csv
- boroughs.csv
- schedules.xml

## Part 3 - Load raw datasets

In [0]:
stations_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(f"{raw_base_path}/stations.csv")
)

lines_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(f"{raw_base_path}/lines.csv")
)

vehicle_types_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(f"{raw_base_path}/vehicle_types.csv")
)

operators_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(f"{raw_base_path}/operators.csv")
)

zones_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(f"{raw_base_path}/zones.csv")
)

fares_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(f"{raw_base_path}/fares.csv")
)

boroughs_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(f"{raw_base_path}/boroughs.csv")
)


journeys_df = spark.read.option("multiline", True).json(f"{raw_base_path}/journeys.json")
disruptions_df = spark.read.option("multiline", True).json(f"{raw_base_path}/disruptions.json")


schedules_df = (
    spark.read
    .format("xml")
    .option("rowTag", "schedule")
    .load(f"{raw_base_path}/schedules.xml")
)


# Part 4 - Explore schemas and preview data

In [0]:
stations_df.printSchema()
lines_df.printSchema()
journeys_df.printSchema()

stations_df.show(5, truncate=False)
lines_df.show(5, truncate=False)
journeys_df.show(5, truncate=False)

stations_df.show(5, truncate=False)
lines_df.show(5, truncate=False)
journeys_df.show(5, truncate=False)

print("stations rows:", stations_df.count())
print("lines rows:", lines_df.count())
print("journeys rows:", journeys_df.count())



# Part 5 - Clean and standardize datasets

In [0]:
stations_clean_df = (
    stations_df
    .withColumn("station_id", F.trim(F.col("station_id")))
    .withColumn("station_name", F.initcap(F.trim(F.col("station_name"))))
    .withColumn("borough_id", F.trim(F.col("borough_id")))
    .dropDuplicates(["station_id"])
    .filter(F.col("station_id").isNotNull())
)


lines_clean_df = (
    lines_df
    .withColumn("line_id", F.trim(F.col("line_id")))
    .withColumn("line_name", F.initcap(F.trim(F.col("line_name"))))
    .withColumn("operator_id", F.trim(F.col("operator_id")))
    .withColumn("vehicle_type_id", F.trim(F.col("vehicle_type_id")))
    .dropDuplicates(["line_id"])
    .filter(F.col("line_id").isNotNull())
)


boroughs_clean_df = (
    boroughs_df
    .withColumn("borough_id", F.trim(F.col("borough_id")))
    .withColumn("borough_name", F.initcap(F.trim(F.col("borough_name"))))
    .dropDuplicates(["borough_id"])
    .filter(F.col("borough_id").isNotNull())
)

zones_clean_df = (
    zones_df
    .withColumn("zone_id", F.trim(F.col("zone_id")))
    .dropDuplicates(["zone_id"])
    .filter(F.col("zone_id").isNotNull())
)

journeys_clean_df = (
    journeys_df
    .withColumn("journey_id", F.trim(F.col("journey_id")))
    .withColumn("line_id", F.trim(F.col("line_id")))
    .withColumn("station_id", F.trim(F.col("station_id")))
    .withColumn("passenger_count", F.expr("try_cast(passenger_count as int)"))
    .withColumn("delay_minutes", F.expr("try_cast(delay_minutes as int)"))
    .dropDuplicates(["journey_id"])
    .filter(F.col("journey_id").isNotNull())
    .filter(F.col("passenger_count").isNotNull())
    .filter(F.col("passenger_count") >= 0)
    .filter(F.col("delay_minutes").isNotNull())
    .filter(F.col("delay_minutes") >= 0)
)

stations_clean_df.show(5, truncate=False)
journeys_clean_df.show(5, truncate=False)

print("clean stations:", stations_clean_df.count())
print("clean journeys:", journeys_clean_df.count())

In [0]:
stations_clean_df.show(5, truncate=False)
journeys_clean_df.show(5, truncate=False)

print("clean stations:", stations_clean_df.count())
print("clean journeys:", journeys_clean_df.count())

# Part 6 - Join and transform datasets

In [0]:
transport_activity_df = (
    journeys_clean_df.alias("j")
    .join(lines_clean_df.alias("l"), F.col("j.line_id") == F.col("l.line_id"), "left")
    .join(stations_clean_df.alias("s"), F.col("j.station_id") == F.col("s.station_id"), "left")
    .join(boroughs_clean_df.alias("b"), F.col("s.borough_id") == F.col("b.borough_id"), "left")
    .select(
        F.col("j.journey_id"),
        F.col("j.line_id"),
        F.col("l.line_name"),
        F.col("j.station_id"),
        F.col("s.station_name"),
        F.col("b.borough_name"),
        F.col("j.passenger_count"),
        F.col("j.delay_minutes")
    )
)

In [0]:
line_operations_df = (
    lines_clean_df.alias("l")
    .join(operators_df.alias("o"), F.col("l.operator_id") == F.col("o.operator_id"), "left")
    .join(vehicle_types_df.alias("v"), F.col("l.vehicle_type_id") == F.col("v.vehicle_type_id"), "left")
    .select(
        F.col("l.line_id"),
        F.col("l.line_name"),
        F.col("o.operator_name"),
        F.col("v.vehicle_type_name")
    )
)

In [0]:
disruptions_by_line_df = (
    disruptions_df.alias("d")
    .join(lines_clean_df.alias("l"), F.col("d.line_id") == F.col("l.line_id"), "left")
    .select(
        F.col("d.disruption_id"),
        F.col("d.line_id"),
        F.col("l.line_name"),
        F.col("d.disruption_type"),
        F.col("l.service_status")
    )
)

In [0]:
fare_summary_df = (
    fares_df.alias("f")
    .join(zones_clean_df.alias("z"), F.col("f.zone_id") == F.col("z.zone_id"), "left")
    .select(
        F.col("f.fare_id"),
        F.col("f.zone_id"),
        F.col("z.zone_name"),
        F.col("f.fare_amount")
    )
)

# Part 7 - Create processed outputs

In [0]:
top_stations_df = (
    transport_activity_df
    .groupBy("station_id", "station_name")
    .agg(F.sum("passenger_count").alias("total_passengers"))
    .orderBy(F.desc("total_passengers"))
)

In [0]:
line_delay_report_df = (
    transport_activity_df
    .groupBy("line_id", "line_name")
    .agg(F.avg("delay_minutes").alias("average_delay_minutes"))
    .orderBy(F.desc("average_delay_minutes"))
)

In [0]:
borough_passenger_report_df = (
    transport_activity_df
    .groupBy("borough_name")
    .agg(F.sum("passenger_count").alias("total_passengers"))
    .orderBy(F.desc("total_passengers"))
)

In [0]:
disruption_summary_df = (
    disruptions_by_line_df
    .groupBy("disruption_type")
    .agg(F.count("*").alias("disruption_count"))
    .orderBy(F.desc("disruption_count"))
)

In [0]:
line_passenger_report_df = (
    transport_activity_df
    .groupBy("line_id", "line_name")
    .agg(F.sum("passenger_count").alias("total_passengers"))
    .orderBy(F.desc("total_passengers"))
)

# Part 8 - Create curated business table

In [0]:
top_stations_df.write.mode("overwrite").saveAsTable("workspace.default.top_stations")

In [0]:
top_stations_df.write.mode("overwrite").saveAsTable("top_stations")
line_delay_report_df.write.mode("overwrite").saveAsTable("line_delay_report")
borough_passenger_report_df.write.mode("overwrite").saveAsTable("borough_passenger_report")
disruption_summary_df.write.mode("overwrite").saveAsTable("disruption_summary")
line_passenger_report_df.write.mode("overwrite").saveAsTable("line_passenger_report")

#Part 9 - Validate with SQL

In [0]:
%sql
SHOW TABLES;

In [0]:
%sql
SELECT *
FROM top_stations
LIMIT 10;

In [0]:
%sql
SELECT COUNT(*) AS row_count
FROM top_stations;

In [0]:
%sql
SELECT station_name, total_passengers
FROM top_stations
ORDER BY total_passengers DESC
LIMIT 10;

In [0]:
%sql
SELECT line_name, average_delay_minutes
FROM line_delay_report
ORDER BY average_delay_minutes DESC
LIMIT 10;

In [0]:
%sql
SELECT borough_name, total_passengers
FROM borough_passenger_report
ORDER BY total_passengers DESC
LIMIT 10;